# Floor-marker reader v3: clean images in, damage made here (YOLO-pose)

You upload only **clean** marker images (labelled automatically from their tags) and the augmentation scripts. This notebook then creates the damaged training images itself: tape, wiped frame sides, dust film, specks, smears, faded print, scratches, glare, blur, and **random AprilTag codes** in 70 % of the images so the model learns "four tags in a red frame", not particular stations.

The model has **one class, `marker`**, and 20 keypoints (4 frame corners, 16 tag corners) in **image order**: the corner nearest the picture's top-left first, then clockwise (v2 mixed up corners because the marker looks the same turned by 90°). The clean images are lit by the robot's own **LED ring**. No rotation augmentation here (`degrees=0`): it would break the image order. The station is read from the tags on the robot, so new stations need no retraining.

Against overfitting: every damage is generated afresh (nothing repeats); all variants of one clean image stay in the same split; nano model, early stopping; train vs validation losses plotted; a held-out test set (other poses, unseen damage colours/shapes, floor mats and light) scored once at the end.

**Steps**: Runtime > Change runtime type > **T4 GPU**; run the cells in order; upload `markers_v3_clean.zip`, `markers_v3_clean_test.zip` and `marker_augment.zip`. The last cell downloads `marker_reader_v3.zip`; put it in the workspace's `ai_data/` folder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q ultralytics==8.3.40 onnx onnxslim onnxscript

In [ ]:
# Save the training to Google Drive: a disconnect then loses nothing.
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/tugger_runs'
import os; os.makedirs(RUNS, exist_ok=True)


In [ ]:
# The zips are read from Google Drive (MyDrive/tugger_data) if they are there;
# otherwise you are asked to upload them once, and they are kept in Drive.
import os, shutil
from google.colab import files
DATA = '/content/drive/MyDrive/tugger_data'
os.makedirs(DATA, exist_ok=True)
NEEDED = ['markers_v3_clean.zip', 'markers_v3_clean_test.zip', 'marker_augment.zip']
missing = [z for z in NEEDED if not os.path.exists(f'{DATA}/{z}')]
if missing:
    print('Upload:', missing)
    for z in files.upload():
        shutil.move(z, f'{DATA}/{z}')
!unzip -q -o {DATA}/markers_v3_clean.zip -d /content/
!unzip -q -o {DATA}/markers_v3_clean_test.zip -d /content/
!unzip -q -o {DATA}/marker_augment.zip -d /content/aug
!ls /content

In [ ]:
# Damaged training images from the clean ones (a few minutes).
%cd /content/aug
!python3 augment.py --clean /content/markers_v3_clean --out /content/markers_v3 --variants 3 --seed 5
!python3 augment.py --clean /content/markers_v3_clean_test --out /content/markers_v3_test --variants 3 --seed 97 --test
%cd /content
!ls /content/markers_v3/images/train | wc -l; ls /content/markers_v3/images/val | wc -l; ls /content/markers_v3_test/images/test | wc -l

In [ ]:
import yaml
cfg = yaml.safe_load(open('/content/markers_v3/data.yaml'))
cfg['path'] = '/content/markers_v3'
yaml.safe_dump(cfg, open('/content/markers_v3/data_colab.yaml', 'w'))
test = dict(cfg, path='/content/markers_v3_test', train='images/test', val='images/test')
yaml.safe_dump(test, open('/content/markers_v3_test/data_test.yaml', 'w'))
cfg

In [ ]:
import numpy as np
np.trapz = np.trapezoid   # numpy 2 renamed it; Ultralytics 8.3.40 still calls the old name
from ultralytics import YOLO
model = YOLO('yolo11n-pose.pt')
model.train(data='/content/markers_v3/data_colab.yaml', imgsz=640, epochs=80, batch=32,
            patience=20, fliplr=0.0, flipud=0.0, degrees=0.0, mosaic=0.5, scale=0.3,
            translate=0.1, hsv_h=0.01, hsv_s=0.4, hsv_v=0.5, weight_decay=0.0005,
            project=RUNS, name='markers_v3', seed=11, deterministic=True)

In [ ]:
# ONLY if the training stopped (disconnect, restart): run cells 1, 2 (Drive) and this one;
# it continues from the last finished epoch.
import numpy as np
np.trapz = np.trapezoid
from ultralytics import YOLO
YOLO(f'{RUNS}/markers_v3/weights/last.pt').train(resume=True)


In [ ]:
# Overfitting check: training and validation losses side by side.
import pandas as pd, matplotlib.pyplot as plt
RUNS = '/content/drive/MyDrive/tugger_runs'   # also works after a restart (run cell 2 first)
r = pd.read_csv(f'{RUNS}/markers_v3/results.csv')
r.columns = [c.strip() for c in r.columns]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, name in zip(axes, ['pose_loss', 'box_loss']):
    ax.plot(r['epoch'], r[f'train/{name}'], label='train')
    ax.plot(r['epoch'], r[f'val/{name}'], label='validation')
    ax.set_title(name); ax.set_xlabel('epoch'); ax.legend()
plt.tight_layout(); plt.savefig(f'{RUNS}/markers_v3/overfitting_check.png', dpi=150); plt.show()

In [ ]:
best = YOLO(f'{RUNS}/markers_v3/weights/best.pt')
val = best.val(data='/content/markers_v3/data_colab.yaml', imgsz=640, project=RUNS, name='v3_val')
test = best.val(data='/content/markers_v3_test/data_test.yaml', imgsz=640, project=RUNS, name='v3_heldout_test')
summary = {'val_pose_mAP50-95': float(val.pose.map), 'val_box_mAP50-95': float(val.box.map),
           'test_pose_mAP50-95': float(test.pose.map), 'test_box_mAP50-95': float(test.box.map)}
print(summary)
import json; json.dump(summary, open(f'{RUNS}/markers_v3/summary.json', 'w'), indent=1)
best.export(format='onnx', imgsz=640, opset=12, simplify=True)

In [ ]:
!cd {RUNS} && zip -q -r /content/marker_reader_v3.zip markers_v3 v3_val v3_heldout_test -x '*/weights/last.pt'
from google.colab import files
files.download('/content/marker_reader_v3.zip')